# Assignment 6

Sara Milovanova, Biljana Vitanova

In [ ]:
import pandas as pd
import time
from confluent_kafka import Consumer
from config import *
import json

In [ ]:
TOP_N = 10
BOOTSTRAP = "localhost:10000,localhost:10001"

TOPIC_RAW = "yellow-taxi-events"
TOPIC_BOROUGH_STATS = "taxi-borough-stats"
TOPIC_LOCATION_STATS = "taxi-location-stats"
TOPIC_CLUSTERS = "taxi-clusters"

WINDOW_SECONDS = 30
FIELDS = ['trip_distance', 'fare_amount', 'tip_amount']  # 3+ attributes for stats -> also pickup/dropoff business counts?

In [ ]:
from confluent_kafka.admin import (
    AdminClient,
    NewTopic
)

from config import *

admin = AdminClient({
    "bootstrap.servers": BOOTSTRAP
})

TOPICS = [
    TOPIC_RAW,
    TOPIC_BOROUGH_STATS,
    TOPIC_LOCATION_STATS,
    TOPIC_CLUSTERS
]

print("Deleting topics...")

admin.delete_topics(TOPICS)

time.sleep(3)

print("Recreating topics...")

new_topics = [
    NewTopic(
        topic=t,
        num_partitions=1,
        replication_factor=1
    )
    for t in TOPICS
]

admin.create_topics(new_topics)

print("Topics reset complete.")

In [ ]:
# from confluent_kafka.admin import AdminClient, NewTopic

# admin = AdminClient({'bootstrap.servers': 'localhost:10000,localhost:10001'})

# TOPICS = ['yellow-taxi-events', 'taxi-borough-stats', 'taxi-location-stats', 'taxi-clusters']

# # Delete existing
# admin.delete_topics(TOPICS)

# import time; time.sleep(2)  # give Kafka a moment

# # Recreate
# new_topics = [NewTopic(t, num_partitions=1, replication_factor=1) for t in TOPICS]
# admin.create_topics(new_topics)
# print("Topics reset.")

Topics reset.


## Data Preview

In [ ]:
df = pd.read_parquet('september_first_two_days.parquet')

In [8]:
df.dtypes

tpep_pickup_datetime      datetime64[ns]
tpep_dropoff_datetime     datetime64[ns]
PULocationID                       int64
DOLocationID                       int64
PU_Borough                           str
PU_Zone                              str
DO_Borough                           str
DO_Zone                              str
trip_distance                    float64
fare_amount                      float64
tip_amount                       float64
total_amount                     float64
pickup_time_hour          datetime64[ns]
temperature_c                    float64
precipitation_mm                 float64
pickup_business_count              int64
dropoff_business_count             int64
dtype: object

In [13]:
df.head(10)

,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,PU_Borough,PU_Zone,DO_Borough,DO_Zone,trip_distance,fare_amount,tip_amount,total_amount,pickup_time_hour,temperature_c,precipitation_mm,pickup_business_count,dropoff_business_count
0,2019-09-01 00:00:00,2019-09-01 00:17:16,229,112,Manhattan,Sutton Place/Turtle Bay North,Brooklyn,Greenpoint,4.19,16.0,1.50,21.30,2019-09-01,23.5,0.0,193,285
1,2019-09-01 00:00:00,2019-09-01 00:05:47,148,256,Manhattan,Lower East Side,Brooklyn,Williamsburg (South Side),2.00,8.0,2.00,13.80,2019-09-01,23.5,0.0,223,181
2,2019-09-01 00:00:00,2019-09-01 00:26:19,256,37,Brooklyn,Williamsburg (South Side),Brooklyn,Bushwick South,3.82,17.5,3.76,24.51,2019-09-01,23.5,0.0,181,323
3,2019-09-01 00:00:00,2019-09-01 00:05:26,231,261,Manhattan,TriBeCa/Civic Center,Manhattan,World Trade Center,1.16,6.0,0.00,9.80,2019-09-01,23.5,0.0,205,77
4,2019-09-01 00:00:00,2019-09-01 00:33:02,74,226,Manhattan,East Harlem North,Queens,Sunnyside,10.03,34.0,0.00,41.42,2019-09-01,23.5,0.0,288,357
5,2019-09-01 00:00:00,2019-09-01 00:27:46,114,80,Manhattan,Greenwich Village South,Brooklyn,East Williamsburg,5.10,22.0,0.00,25.80,2019-09-01,23.5,0.0,77,251
6,2019-09-01 00:00:01,2019-09-01 00:19:42,211,230,Manhattan,SoHo,Manhattan,Times Sq/Theatre District,2.90,14.5,3.65,21.95,2019-09-01,23.5,0.0,77,248
7,2019-09-01 00:00:01,2019-09-01 00:17:03,132,95,Queens,JFK Airport,Queens,Forest Hills,8.40,25.0,0.00,26.30,2019-09-01,23.5,0.0,20,369
8,2019-09-01 00:00:01,2019-09-01 00:04:02,79,148,Manhattan,East Village,Manhattan,Lower East Side,0.40,4.5,1.65,9.95,2019-09-01,23.5,0.0,256,223
9,2019-09-01 00:00:02,2019-09-01 00:21:44,181,229,Brooklyn,Park Slope,Manhattan,Sutton Place/Turtle Bay North,7.98,25.0,5.76,34.56,2019-09-01,23.5,0.0,255,193


In [14]:
df.shape

(273852, 17)

In [15]:
df.describe()

,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,tip_amount,total_amount,pickup_time_hour,temperature_c,precipitation_mm,pickup_business_count,dropoff_business_count
count,273852,273852,273852.000000,273852.000000,273852.000000,273852.000000,273852.000000,273852.000000,273852,273852.000000,273852.000000,273852.000000,273852.000000
mean,2019-09-02 01:36:48.728992512,2019-09-02 01:53:33.563355136,156.623231,153.882608,3.452271,13.460691,2.041913,19.139523,2019-09-02 01:06:34.767976704,22.155850,0.270082,176.660094,188.125341
min,2019-09-01 00:00:00,2019-09-01 00:01:06,4.000000,4.000000,0.000000,0.000000,0.000000,0.000000,2019-09-01 00:00:00,17.300000,0.000000,0.000000,0.000000
25%,2019-09-01 14:37:07.750000128,2019-09-01 14:52:52.750000128,113.000000,97.000000,1.070000,6.500000,0.000000,10.800000,2019-09-01 14:00:00,20.600000,0.000000,113.000000,128.000000
50%,2019-09-01 23:06:20,2019-09-01 23:23:08,158.000000,158.000000,1.820000,9.000000,1.580000,14.065000,2019-09-01 23:00:00,22.800000,0.000000,178.000000,178.000000
75%,2019-09-02 14:41:05,2019-09-02 14:54:49.249999872,230.000000,230.000000,3.600000,15.000000,2.700000,20.300000,2019-09-02 14:00:00,23.700000,0.000000,220.000000,223.000000
max,2019-09-02 23:59:59,2019-09-30 06:45:51,263.000000,263.000000,79.500000,423.000000,280.000000,837.930000,2019-09-02 23:00:00,25.600000,3.600000,693.000000,693.000000
std,NaN,NaN,64.957638,70.895204,4.300726,12.122381,2.819658,14.821147,NaN,2.230242,0.790346,109.805671,110.544152


In [ ]:
event_columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",

    "PULocationID",
    "DOLocationID",

    "PU_Borough",
    "PU_Zone",

    "DO_Borough",
    "DO_Zone",

    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",

    "temperature_c",
    "precipitation_mm",

    "pickup_business_count",
    "dropoff_business_count"
]

Before streaming, check the **top 10 locations** offline.

In [19]:
df = df.sort_values('tpep_pickup_datetime').reset_index(drop=True)

# Combined pickups + dropoffs per location
combined = df['PULocationID'].value_counts().add(
    df['DOLocationID'].value_counts(), fill_value=0
)
TOP10_LOCATIONS = combined.nlargest(TOP_N).index.tolist()
print("Top 10 locations:", TOP10_LOCATIONS)

# Map these locations to their boroughs and zones
location_info = df[['PULocationID', 'PU_Borough', 'PU_Zone']].drop_duplicates().set_index('PULocationID')
top_locations_info = location_info.loc[TOP10_LOCATIONS]
print("\nTop 10 locations with borough and zone info:")
print(top_locations_info)

Top 10 locations: [186, 48, 132, 230, 161, 170, 79, 68, 236, 162]

Top 10 locations with borough and zone info:
             PU_Borough                       PU_Zone
PULocationID                                         
186           Manhattan  Penn Station/Madison Sq West
48            Manhattan                  Clinton East
132              Queens                   JFK Airport
230           Manhattan     Times Sq/Theatre District
161           Manhattan                Midtown Center
170           Manhattan                   Murray Hill
79            Manhattan                  East Village
68            Manhattan                  East Chelsea
236           Manhattan         Upper East Side North
162           Manhattan                  Midtown East


## Architecture / Setup description 

In [ ]:
# producer script code i used (to run in terminal, not in notebook)

In [ ]:
# consumer script code i used (to run in terminal, not in notebook)

In [ ]:
# Quix Streams code (to run in terminal, not in notebook)

## Results 
- consume from the stats topics and display DataFrames + plots

In [ ]:
def consume_stats(topic, n=50):

    c = Consumer({
        "bootstrap.servers": BOOTSTRAP,
        "group.id": f"{topic}-reader",
        "auto.offset.reset": "earliest"
    })

    c.subscribe([topic])

    results = []

    deadline = time.time() + 30

    while len(results) < n and time.time() < deadline:

        msg = c.poll(1.0)

        if msg is None:
            continue

        if msg.error():
            continue

        results.append(
            json.loads(
                msg.value().decode("utf-8")
            )
        )

    c.close()

    return pd.DataFrame(results)

In [ ]:
borough_stats_df = consume_stats(
    TOPIC_BOROUGH_STATS
)

display(borough_stats_df)

In [ ]:
location_stats_df = consume_stats(
    TOPIC_LOCATION_STATS
)

display(location_stats_df)

## Clustering 
- show the River code, display cluster assignments and scatter plots